# HW2.5 — Synthetic data via SD + ControlNet: visualization

Просмотр сгенерированной синтетики и итогов ablation.  
Генерация запускается через `scripts/generate_synth.sh`, обучение классификатора — `scripts/run_classifier_ablation.sh`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## 1. Примеры синтетики по классам

In [ ]:
SYNTH = Path('../data/synth')
for cls_dir in sorted(SYNTH.iterdir()):
    if not cls_dir.is_dir():
        continue
    samples = sorted(cls_dir.glob('*.png'))[:4]
    if not samples:
        continue
    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
    if len(samples) == 1:
        axes = [axes]
    for ax, p in zip(axes, samples):
        ax.imshow(Image.open(p))
        ax.set_title(p.name, fontsize=8)
        ax.axis('off')
    fig.suptitle(f'class: {cls_dir.name}')
    plt.tight_layout()
    plt.show()

## 2. Canny-приоры (что подаём в ControlNet)

In [ ]:
demo = Path('../assets/canny_priors_demo.png')
if demo.exists():
    plt.figure(figsize=(10, 5))
    plt.imshow(Image.open(demo)); plt.axis('off')
    plt.title('Procedural Canny priors для stop_sign / traffic_light')
    plt.show()

## 3. Ablation-таблица: реал vs +25% / +50% / +100% синтетики

In [ ]:
import pandas as pd

RES = Path('../runs/classifier')
rows = []
for run in sorted(RES.iterdir()):
    js = run / 'metrics.json'
    if js.exists():
        d = json.load(open(js))
        d['run'] = run.name
        rows.append(d)
df = pd.DataFrame(rows)
df = df[['run', 'accuracy', 'macro_f1', 'rare_class_recall']]
df

In [ ]:
# bar-плот по rare-class recall
if len(df):
    df.plot.bar(x='run', y=['accuracy', 'macro_f1', 'rare_class_recall'], figsize=(10, 4))
    plt.ylabel('metric'); plt.xticks(rotation=15, ha='right'); plt.grid(axis='y'); plt.show()

## 4. Выводы

Заполняется после прогона — основные ожидания:

- **+25% synth**: умеренный буст редких классов без потери accuracy → лучший trade-off.  
- **+50% synth**: насыщение, редкий класс растёт, общий accuracy стабилен.  
- **+100% synth**: либо плато, либо лёгкая просадка из-за domain shift синтетики.  

См. развёрнутый разбор в `reports/HW2.5_report.md`.